In [1]:
import pandas as pd

nba = pd.read_csv("NBA_stats.csv")
nba.head(n=5)

,Rk,Player,Pos,Age,Tm,G,GS,MP,FG,FGA,...,FT%,ORB,DRB,TRB,AST,STL,BLK,TOV,PF,PTS
0,1,Álex Abrines\abrinal01,SG,23,OKC,68,6,15.5,2.0,5.0,...,0.898,0.3,1.0,1.3,0.6,0.5,0.1,0.5,1.7,6.0
1,2,Quincy Acy\acyqu01,PF,26,TOT,38,1,14.7,1.8,4.5,...,0.750,0.5,2.5,3.0,0.5,0.4,0.4,0.6,1.8,5.8
2,2,Quincy Acy\acyqu01,PF,26,DAL,6,0,8.0,0.8,2.8,...,0.667,0.3,1.0,1.3,0.0,0.0,0.0,0.3,1.5,2.2
3,2,Quincy Acy\acyqu01,PF,26,BRK,32,1,15.9,2.0,4.8,...,0.754,0.6,2.8,3.3,0.6,0.4,0.5,0.6,1.8,6.5
4,3,Steven Adams\adamsst01,C,23,OKC,80,80,29.9,4.7,8.2,...,0.611,3.5,4.2,7.7,1.1,1.1,1.0,1.8,2.4,11.3


In [2]:
wnba = pd.read_csv("WNBA_stats.csv")
wnba.head(n=5)

,Name,Team,Pos,Height,Weight,BMI,Birth_Place,Birthdate,Age,College,...,OREB,DREB,REB,AST,STL,BLK,TO,PTS,DD2,TD3
0,Aerial Powers,DAL,F,183,71.0,21.200991,US,"January 17, 1994",23,Michigan State,...,6,22,28,12,3,6,12,93,0,0
1,Alana Beard,LA,G/F,185,73.0,21.329438,US,"May 14, 1982",35,Duke,...,19,82,101,72,63,13,40,217,0,0
2,Alex Bentley,CON,G,170,69.0,23.875433,US,"October 27, 1990",26,Penn State,...,4,36,40,78,22,3,24,218,0,0
3,Alex Montgomery,SAN,G/F,185,84.0,24.543462,US,"December 11, 1988",28,Georgia Tech,...,35,134,169,65,20,10,38,188,2,0
4,Alexis Jones,MIN,G,175,78.0,25.469388,US,"August 5, 1994",23,Baylor,...,3,9,12,12,7,0,14,50,0,0


## NBA vs WNBA — style of play comparisons

The NBA dataset reports per-game averages; the WNBA dataset reports season totals, so WNBA stats are divided by `Games Played` to put both leagues on a per-game scale. Field goal and three-point percentages are rescaled to 0–100 so both leagues share a common axis.

### introduction
The NBA (National Basketball Association) and the WNBA (Women's National Basketball Association) represent the highest level of professional basketball competition in the United States. Both leagues operate under largely the same ruleset, share the same positions and terminology, and are played on the same court. Despite these similarities, the two leagues have developed independently over decades. Each league is competing only against teams within its own league, and cultivating its own competitive ecosystem. This raises a natural question: do the NBA and WNBA actually play basketball differently, or are the observable differences superficial? driven by factors like playing time and roster size rather than genuine differences in style? 

This data story attempts to answer that question by analyzing and visualizing per-game player statistics from both leagues, focusing on shooting, rebounding, and defensive output. Rather than looking at surrounding factors such as team budgets, audience size, or physical attributes, we focus entirely on the statistical patterns that emerge from the games themselves. 

### Perspectives and arguments 
The analysis is structured around two competing perspectives. 

#### Perspective 1: 
The NBA and WNBA have meaningfully different styles of play. From this perspective, the statistical distributions of the two leagues reflect genuinely different approaches to the game. Not just differences in the scale of the numbers, but differences in how the game is played tactically and physically. The central argument is that the WNBA exhibits a more controlled, defensively oriented style, while the NBA is characterized by a more aggressive, offensive approach. 


#### Perspective 2: 
The fundamental structure of play is the same across both leagues. From this perspective, the differences in raw statistics reflect differences in the scale and intensity of play, influenced by factors such as game pace and roster composition, rather than a fundamentally different style. The argument is that once you look at how statistics relate to each other (rather than their absolute values), the two leagues follow the same underlying patterns. 

### Dataset and Preprocessing

This data story uses two publicly available datasets of per-game player statistics: [one for the NBA](https://www.kaggle.com/datasets/abdurahmanmaarouf/nba-players-stats-2016-2017?select=NBA+Players+Stats+201617.csv) and [one for the WNBA](https://www.kaggle.com/datasets/jinxbe/wnba-player-stats-2017). The NBA dataset reports per-game averages directly; the WNBA dataset reports season totals, which were divided by games played during preprocessing to bring both datasets onto the same per-game scale. Field goal and three point percentages were rescaled to a 0–100 range so both leagues share a common axis in comparative visualizations. Players with missing values in the relevant columns were excluded from the specific visualizations that depend on those columns, but were retained in the dataset otherwise. 

All variables were changed to have the same name, and both datasets were then combined into a dataset under a variable called league, which can either be "NBA" or "WNBA".

The variables used across the visualizations include: field goals attempted (FGA), field goal percentage (FG%), three-point attempts (3PA), points per game (PTS), offensive rebounds (ORB), defensive rebounds (DRB), total rebounds (TRB), assists (AST), turnovers (TOV), steals (STL), blocks (BLK), and minutes played per game (MP). 

In [3]:
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy.stats import gaussian_kde
import altair as alt
from IPython.display import display, HTML

pio.templates.default = "plotly_white"
pio.renderers.default = "notebook_connected"


# Original league pair — kept for the FGA violin (chart 1)
LEAGUE_COLORS = {"NBA": "#66c2a5", "WNBA": "#fc8d62"}
ALT_LEAGUE_COLORS = alt.Scale(domain=['NBA', 'WNBA'], range=['#66c2a5', '#fc8d62'])

# Per-chart palettes so each plot has its own NBA / WNBA contrast
PALETTE_DENSITY  = alt.Scale(domain=['NBA', 'WNBA'], range=['#5e60ce', '#ff6b6b'])
PALETTE_3PA      = alt.Scale(domain=['NBA', 'WNBA'], range=['#2d6a4f', '#e9c46a'])
PALETTE_ECDF     = alt.Scale(domain=['NBA', 'WNBA'], range=['#3a86ff', '#fb5607'])
PALETTE_STRIP    = alt.Scale(domain=['NBA', 'WNBA'], range=['#3d5a80', '#ee6c4d'])
PALETTE_BUBBLE   = alt.Scale(domain=['NBA', 'WNBA'], range=['#7209b7', '#52b788'])
PALETTE_DUMBBELL = {"NBA": "#9d0208", "WNBA": "#ffba08"}
PALETTE_BOX      = {"NBA": "#1d3557", "WNBA": "#e76f51"}
PALETTE_KDE      = alt.Scale(domain=['NBA', 'WNBA'], range=['#118ab2', '#ffd166'])

BASE_FONT = dict(family="Arial", size=13)

alt.renderers.enable('html')

def show_altair(chart):
    display(chart)

# Render Plotly figures via direct Plotly.newPlot calls (no RequireJS),
# so they work in the Jupyter Book / Sphinx HTML output.
def show_plotly(fig):
    display(HTML(pio.to_html(fig, include_plotlyjs=False, full_html=False)))

def pos_bucket(p):
    if p in ('PG', 'SG', 'G'): return 'Guard'
    if p in ('SF', 'PF', 'F'): return 'Forward'
    if p == 'C': return 'Center'
    return 'Hybrid'

# NBA-only: when a player was traded mid-season the CSV has one row per
# team plus a TOT (season total) row. Keep just the TOT row for those
# players so every player ends up as a single averaged row.
_with_tot = set(nba.loc[nba['Tm'] == 'TOT', 'Player'])
nba = nba[(nba['Tm'] == 'TOT') | (~nba['Player'].isin(_with_tot))].copy()

nba_p = nba[['Player', 'Pos', 'MP', 'FGA', 'FG', '3PA', '3P%', 'FG%', 'PTS',
             'AST', 'STL', 'BLK', 'TOV', 'ORB', 'DRB', 'TRB']].copy()
nba_p['Player'] = nba_p['Player'].str.split('\\').str[0]
nba_p['FG%'] = nba_p['FG%'] * 100
nba_p['3P%'] = nba_p['3P%'] * 100
nba_p['League'] = 'NBA'

wnba_p = pd.DataFrame({
    'Player': wnba['Name'],
    'Pos':    wnba['Pos'],
    'MP':  wnba['MIN']  / wnba['Games Played'],
    'FGA': wnba['FGA']  / wnba['Games Played'],
    'FG':  wnba['FGM']  / wnba['Games Played'],
    '3PA': wnba['3PA']  / wnba['Games Played'],
    '3P%': wnba['3P%'],
    'FG%': wnba['FG%'],
    'PTS': wnba['PTS']  / wnba['Games Played'],
    'AST': wnba['AST']  / wnba['Games Played'],
    'STL': wnba['STL']  / wnba['Games Played'],
    'BLK': wnba['BLK']  / wnba['Games Played'],
    'TOV': wnba['TO']   / wnba['Games Played'],
    'ORB': wnba['OREB'] / wnba['Games Played'],
    'DRB': wnba['DREB'] / wnba['Games Played'],
    'TRB': wnba['REB']  / wnba['Games Played'],
})
wnba_p['League'] = 'WNBA'

combined = pd.concat([nba_p, wnba_p], ignore_index=True)
combined['N3PA'] = combined['FGA'] - combined['3PA']
combined['PosGroup'] = combined['Pos'].apply(pos_bucket)
combined['ORB_share'] = combined['ORB'] / combined['TRB'].replace(0, pd.NA)
combined.groupby('League')[['MP','FGA','PTS','3PA','FG%','AST','STL','BLK','TOV','ORB','DRB','TRB']].agg(['mean','std']).round(2).T

League      NBA   WNBA
MP  mean  19.54  19.04
    std    8.84   9.08
FGA mean   6.73   6.40
    std    4.37   3.93
PTS mean   8.19   7.70
    std    5.84   5.23
3PA mean   2.17   1.65
    std    1.89   1.64
FG% mean  44.13  42.90
    std   10.28  10.11
AST mean   1.75   1.69
    std    1.70   1.50
STL mean   0.61   0.67
    std    0.41   0.46
BLK mean   0.39   0.38
    std    0.48   0.46
TOV mean   1.08   1.23
    std    0.77   0.69
ORB mean   0.83   0.84
    std    0.76   0.73
DRB mean   2.67   2.34
    std    1.80   1.65
TRB mean   3.49   3.17
    std    2.43   2.26

### 1. Field goals attempted per game — NBA vs WNBA

When looking at the violin plot below two of the most noteable differences between the NBA and WNBA distributions, are the longer, higher volume tail in the NBA distribution and the tighter clustering of values in the WNBA distribution. The higher volume tail suggests that in the NBA there are more games in which the field goals attempted reach substantually higher values. This indicates a bigger variability in offensiveness between games in te NBA when compared to the WNBA. in the WNBA where the amount of field goals attempted stays quite stable across the different games.

This could suggest a faster pace or aggressiveness in NBA or at least a greater difference in playstyle between different matches in the NBA. In contrast, the WNBA has a stable amount of field goals attempted each game. This could indicates a more stable, controlled pace between matches in comparison to the NBA.

In [4]:
fig = px.violin(
    combined, x='League', y='FGA', color='League',
    box=True, points='all',
    color_discrete_map=LEAGUE_COLORS,
    hover_data=['Player', 'Pos'],
)
fig.update_traces(meanline_visible=True, marker=dict(size=4, opacity=0.55), jitter=0.4)

means = combined.groupby('League')['FGA'].mean().round(2)
for league, m in means.items():
    fig.add_annotation(
        x=league, y=m, text=f'<b>μ = {m}</b>',
        showarrow=False, yshift=18,
        font=dict(size=13, color='#222'),
        bgcolor='rgba(255,255,255,0.92)', bordercolor='#666', borderwidth=1, borderpad=4,
    )

fig.update_layout(
    title=dict(
        text='<b>Field Goals Attempted per Game by League</b><br>'
             '<sub style="color:#666">NBA carries a longer high-volume tail; WNBA distribution is tighter</sub>',
        x=0.02, xanchor='left',
    ),
    yaxis_title='FGA per game', xaxis_title='',
    height=580, showlegend=False,
    margin=dict(t=100, l=70, r=30, b=40),
    font=BASE_FONT,
)
show_plotly(fig)

### 2. Points per game — layered density with league means

Layered density chart: smoothed density curves of points per game for each league, overlaid with a dashed vertical rule on top marking each league's mean — combining the distribution shape and the central tendency in a single layered mark.

In [5]:
pts_data = combined[['PTS', 'League']].dropna()

density = alt.Chart(pts_data).transform_density(
    'PTS',
    as_=['PTS', 'density'],
    groupby=['League'],
    extent=[0, 35],
    steps=200,
).mark_area(opacity=0.45, line={'strokeWidth': 2}).encode(
    x=alt.X('PTS:Q', title='Points per game',
            axis=alt.Axis(values=[5, 10, 15, 20, 25, 30, 35])),
    y=alt.Y('density:Q', title='Density'),
    color=alt.Color('League:N', scale=PALETTE_DENSITY),
)

means_pts = combined.groupby('League', as_index=False)['PTS'].mean()
rules = alt.Chart(means_pts).mark_rule(strokeDash=[4, 2], strokeWidth=2.5).encode(
    x='PTS:Q',
    color=alt.Color('League:N', scale=PALETTE_DENSITY),
    tooltip=['League:N', alt.Tooltip('PTS:Q', title='Mean PTS', format='.2f')],
)

chart_pts = (density + rules).properties(
    width=720, height=400,
    title=alt.TitleParams(
        'Points per Game — Distribution with League Means',
        subtitle='Filled density curves with dashed vertical rule at each league mean',
        anchor='start', fontSize=16,
    ),
).configure_view(stroke=None)

show_altair(chart_pts)

alt.LayerChart(...)

### 3. Three-point attempts vs points scored — interactive linked selection

Scatter of 3-point attempts per game against points scored per game with a draggable rectangular brush — click and drag on the scatter to select a region of players, and the bar chart underneath updates live to show how many of the selected players come from each league.

In [6]:
scatter_data = (
    combined[['3PA', 'PTS', 'Player', 'Pos', 'League']]
    .dropna()
    .rename(columns={'3PA': 'ThreePA'})
)

brush = alt.selection_interval()

scatter = alt.Chart(scatter_data).mark_circle(size=80, opacity=0.55).encode(
    x=alt.X('ThreePA:Q', title='3-point attempts per game',
            axis=alt.Axis(values=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10])),
    y=alt.Y('PTS:Q', title='Points per game'),
    color=alt.condition(
        brush,
        alt.Color('League:N', scale=PALETTE_3PA),
        alt.value('lightgray'),
    ),
    tooltip=[
        alt.Tooltip('Player:N'),
        alt.Tooltip('Pos:N'),
        alt.Tooltip('League:N'),
        alt.Tooltip('ThreePA:Q', title='3PA', format='.2f'),
        alt.Tooltip('PTS:Q', format='.2f'),
    ],
).add_params(brush).properties(
    width=720, height=440,
    title=alt.TitleParams(
        '3PA vs Points per Game — Drag to select',
        subtitle='Selected players are summarised in the bar below',
        anchor='start', fontSize=16,
    ),
)

bars = alt.Chart(scatter_data).mark_bar().encode(
    y=alt.Y('League:N', title=None),
    x=alt.X('count():Q', title='Players inside selection'),
    color=alt.Color('League:N', scale=PALETTE_3PA, legend=None),
    tooltip=['League:N', alt.Tooltip('count():Q', title='Selected players')],
).transform_filter(brush).properties(width=720, height=90)

chart_3pa = (scatter & bars).configure_view(stroke=None)
show_altair(chart_3pa)

alt.VConcatChart(...)

### 4. Field goal percentage — cumulative distribution

Although field goal attempts can indicate a more aggressive or fast paced playstyle, more attempts doesn't necessarily translate to more goals. If a goal is unsuccesfull, the player does not only fail a chance to score, the player also opens up an opportunity for the opposition to claim possesion over the ball. To better understand the decision making involved in the field goal attempts, we plot the field goal percentage, which is the percentage of the field goal attempts that resulted in a goal.

The WNBA shows more players with a below 45 percent field goal percentage than the NBA. A lower field goal percentage may indicate either strong defensive play, which forces low quality shots, or differences in offensive tactics such as pace or reliance on high difficulty attempts. As a result, the field goal percentage needs to be viewed alongside other distributions. 

In [7]:
ecdf_data = combined[combined['FG%'].between(1, 99)][['FG%', 'League']].copy()

chart_ecdf = alt.Chart(ecdf_data).transform_window(
    cumulative_count='count()',
    sort=[{'field': 'FG%'}],
    groupby=['League'],
).transform_joinaggregate(
    total='count()',
    groupby=['League'],
).transform_calculate(
    cum_pct='datum.cumulative_count / datum.total',
).mark_line(strokeWidth=3, interpolate='step-after').encode(
    x=alt.X('FG%:Q', title='FG%'),
    y=alt.Y('cum_pct:Q', title='Share of players at or below',
            axis=alt.Axis(format='.0%')),
    color=alt.Color('League:N', scale=PALETTE_ECDF),
    tooltip=['League:N',
             alt.Tooltip('FG%:Q', format='.1f'),
             alt.Tooltip('cum_pct:Q', title='Share ≤', format='.1%')],
).properties(
    width=760, height=440,
    title=alt.TitleParams(
        'Cumulative Distribution of Field Goal Percentage',
        subtitle='Curves further to the right = higher-shooting league',
        anchor='start', fontSize=16,
    ),
).configure_view(stroke=None)

show_altair(chart_ecdf)

alt.Chart(...)

### 5. Risky shots - Percentage of three pointers attempts made 

One distribution we could look at to further analyze the lower Field goal percentage, is the percentage of three point attempts made, since three pointers are generally a higher risk way to generate a higher score. A higher reliance on three point shots could thus create a lower field goal percentage. As shown by the percentages below however, the WNBA does not attempt these higher difficulty shots as much as the NBA. This suggests that the lower field goal percentage is coming from a higher amount of three point attempts. We can thus conclude that it is more likely that the WNBA has a more defensive playstyle, where offensive plays are more often forced to make an attempt at a goal, even if it does not succed.

In [8]:
tp_rows = []
for league in ['NBA', 'WNBA']:
    sub = combined[combined['League'] == league]
    total = float(sub['3PA'].sum() + sub['N3PA'].sum())
    tp_rows.append({'League': league, 'Type': 'Three point attempts',
                    'Count': float(sub['3PA'].sum()),
                    'Pct': float(sub['3PA'].sum()) / total})
    tp_rows.append({'League': league, 'Type': 'Non three point attempts',
                    'Count': float(sub['N3PA'].sum()),
                    'Pct': float(sub['N3PA'].sum()) / total})
tp_df = pd.DataFrame(tp_rows)

TP_COLORS = alt.Scale(
    domain=['Three point attempts', 'Non three point attempts'],
    range=['#7209b7', '#f9c74f'],
)

def make_threepoint_pie(league_name):
    sub = tp_df[tp_df['League'] == league_name]
    return alt.Chart(sub).mark_arc(
        innerRadius=70, outerRadius=140, stroke='white', strokeWidth=2,
    ).encode(
        theta=alt.Theta('Count:Q', stack=True),
        color=alt.Color('Type:N', scale=TP_COLORS, title=None),
        tooltip=['League:N', 'Type:N',
                 alt.Tooltip('Count:Q', format='.0f'),
                 alt.Tooltip('Pct:Q', title='Share', format='.1%')],
    ).properties(
        width=280, height=320,
        title=alt.TitleParams(league_name, anchor='middle',
                              fontSize=15, fontWeight='bold'),
    )

chart_3pp = alt.hconcat(
    make_threepoint_pie('NBA'), make_threepoint_pie('WNBA'),
).resolve_scale(color='shared').properties(
    title=alt.TitleParams(
        'Three pointer percentage by league',
        subtitle='Each ring shows the share of three point attempts among all field goal attempts',
        anchor='start', fontSize=16,
    ),
).configure_view(stroke=None)

show_altair(chart_3pp)

alt.HConcatChart(...)

## Rebound analysis

A **rebound** is the act of recovering the ball after a missed shot. Every rebound is classified as one of two types:

- **ORB (Offensive Rebound)** — the rebounding team had just taken the shot, so they reclaim possession for a second-chance scoring opportunity.
- **DRB (Defensive Rebound)** — the rebounding team had been defending, so collecting the rebound ends the opponent's possession and lets the team push into offence.
- **TRB (Total Rebounds)** = ORB + DRB, the total number of rebounds a player records per game.

Rebounding is a strong indicator of physical effort, positioning, and the value each league places on second-chance possessions. The next five plots compare these three rebound metrics between the NBA and WNBA across players, positions, and rebound type.

### 6. Total rebounds per game, every player as a point

Horizontal strip plot with one dot per player and a dashed vertical line marking each league's mean, useful for seeing both the typical rebounding rate and the spread of outliers without smoothing the data the way a violin or KDE would.

In [9]:
strip_data = combined[['TRB', 'League', 'Player', 'Pos', 'ORB', 'DRB']].dropna()
means_trb = combined.groupby('League', as_index=False)['TRB'].mean()
means_trb['TRB'] = means_trb['TRB'].round(2)

trb_max = float(strip_data['TRB'].max()) * 1.05
trb_tickvals = [2, 4, 6, 8, 10, 12, 14]

strip = alt.Chart(strip_data).transform_calculate(
    jitter='random()',
).mark_circle(size=55, opacity=0.55, stroke='white', strokeWidth=0.5).encode(
    x=alt.X('TRB:Q', title='TRB per game',
            scale=alt.Scale(domain=[0, trb_max]),
            axis=alt.Axis(values=trb_tickvals)),
    y=alt.Y('League:N', title=None, sort=['NBA', 'WNBA']),
    yOffset='jitter:Q',
    color=alt.Color('League:N', scale=PALETTE_STRIP, legend=None),
    tooltip=['Player:N', 'Pos:N',
             alt.Tooltip('TRB:Q', format='.2f'),
             alt.Tooltip('ORB:Q', format='.2f'),
             alt.Tooltip('DRB:Q', format='.2f')],
)

mean_rules = alt.Chart(means_trb).mark_rule(strokeDash=[4, 2], strokeWidth=2).encode(
    x='TRB:Q',
    color=alt.Color('League:N', scale=PALETTE_STRIP, legend=None),
    tooltip=['League:N', alt.Tooltip('TRB:Q', title='Mean', format='.2f')],
)

chart_trb = (strip + mean_rules).properties(
    width=820, height=380,
    title=alt.TitleParams(
        'Total Rebounds per Game — One Dot per Player',
        subtitle='Dashed lines mark the per-league mean',
        anchor='start', fontSize=16,
    ),
).configure_view(stroke=None)

show_altair(chart_trb)

alt.LayerChart(...)

### 7. Offensive vs defensive rebounds in a bubble chart

Bubble scatter of ORB against DRB per game with marker size encoding total rebounds, so dominant rebounders pop visually and you can spot whether elite WNBA rebounders occupy the same corner of the plane as elite NBA rebounders.

In [10]:
bubble_data = combined[['ORB', 'DRB', 'TRB', 'League', 'Player', 'Pos']].dropna()

chart_bubble = alt.Chart(bubble_data).mark_circle(
    opacity=0.6, stroke='white', strokeWidth=0.5,
).encode(
    x=alt.X('ORB:Q', title='ORB per game',
            axis=alt.Axis(values=[1, 2, 3, 4, 5, 6])),
    y=alt.Y('DRB:Q', title='DRB per game'),
    color=alt.Color('League:N', scale=PALETTE_BUBBLE),
    size=alt.Size('TRB:Q',
                  scale=alt.Scale(range=[20, 500]),
                  title='TRB / game'),
    tooltip=['Player:N', 'Pos:N', 'League:N',
             alt.Tooltip('ORB:Q', format='.2f'),
             alt.Tooltip('DRB:Q', format='.2f'),
             alt.Tooltip('TRB:Q', format='.2f')],
).properties(
    width=820, height=560,
    title=alt.TitleParams(
        'Offensive vs Defensive Rebounds',
        subtitle='Bubble size = total rebounds per game',
        anchor='start', fontSize=16,
    ),
).configure_view(stroke=None)

show_altair(chart_bubble)

alt.Chart(...)

### 8. Mean rebound rate — NBA vs WNBA dumbbell

Dumbbell chart pairing the league means for ORB and DRB and connecting them with a line, where the distance between the two markers is a direct visual of how big the gap is between the leagues on each rebound type.

In [11]:
means_reb = combined.groupby('League')[['ORB', 'DRB']].mean().round(2)
cats = ['ORB', 'DRB']
nba_vals = [means_reb.loc['NBA', c] for c in cats]
wnba_vals = [means_reb.loc['WNBA', c] for c in cats]

fig = go.Figure()
for i, cat in enumerate(cats):
    fig.add_trace(go.Scatter(
        x=[nba_vals[i], wnba_vals[i]], y=[cat, cat],
        mode='lines', line=dict(color='#cccccc', width=5),
        showlegend=False, hoverinfo='skip',
    ))
fig.add_trace(go.Scatter(
    x=nba_vals, y=cats, mode='markers+text', name='NBA',
    marker=dict(size=26, color=PALETTE_DUMBBELL['NBA'], line=dict(color='white', width=2)),
    text=[f'<b>{v}</b>' for v in nba_vals], textposition='top center',
    textfont=dict(size=12, color='#222'),
))
fig.add_trace(go.Scatter(
    x=wnba_vals, y=cats, mode='markers+text', name='WNBA',
    marker=dict(size=26, color=PALETTE_DUMBBELL['WNBA'], line=dict(color='white', width=2)),
    text=[f'<b>{v}</b>' for v in wnba_vals], textposition='bottom center',
    textfont=dict(size=12, color='#222'),
))
fig.update_layout(
    title=dict(
        text='<b>Mean Rebound Rate — NBA vs WNBA</b><br>'
             '<sub style="color:#666">Length of the grey connector = gap between the two leagues</sub>',
        x=0.02, xanchor='left',
    ),
    xaxis_title='Rebounds per game', yaxis_title='',
    height=460,
    margin=dict(t=100, l=80, r=40, b=50),
    font=BASE_FONT,
)
show_plotly(fig)

### 9. Offensive rebound share

Box plot of each player's offensive rebound share (ORB / TRB) by league, which is a style indicator: a higher share suggests a league more willing to crash the offensive glass for second-chance points rather than retreat into transition defence.

In [12]:
share_df = combined.dropna(subset=['ORB_share'])
fig = px.box(
    share_df, x='League', y='ORB_share', color='League',
    color_discrete_map=PALETTE_BOX,
    points='all', hover_data=['Player', 'Pos', 'ORB', 'DRB', 'TRB'],
)
fig.update_traces(marker=dict(size=4, opacity=0.55), jitter=0.4, pointpos=0)

share_means = share_df.groupby('League')['ORB_share'].mean()
for league, m in share_means.items():
    fig.add_annotation(
        x=league, y=m, text=f'<b>μ = {m*100:.0f}%</b>',
        showarrow=False, yshift=18,
        font=dict(size=13, color='#222'),
        bgcolor='rgba(255,255,255,0.92)', bordercolor='#666', borderwidth=1, borderpad=4,
    )

fig.update_layout(
    title=dict(
        text='<b>Offensive Rebound Share (ORB / TRB) by League</b><br>'
             '<sub style="color:#666">Higher share = more willingness to crash the offensive glass</sub>',
        x=0.02, xanchor='left',
    ),
    yaxis_title='ORB / TRB', xaxis_title='',
    height=540, showlegend=False, yaxis_tickformat='.0%',
    margin=dict(t=100, l=70, r=30, b=40),
    font=BASE_FONT,
)
show_plotly(fig)

### 10. Average rebounds by position and league in a heatmap

Heatmap of mean total rebounds per game broken down by position group and league, with each cell carrying its numeric value as an overlaid text label whose colour automatically flips from dark to light on darker cells.

In [13]:
heat_data = (
    combined.groupby(['PosGroup', 'League'], as_index=False)['TRB'].mean()
)
heat_data['TRB'] = heat_data['TRB'].round(2)
pos_sort = ['Guard', 'Forward', 'Hybrid', 'Center']

heat = alt.Chart(heat_data).mark_rect(stroke='white', strokeWidth=2).encode(
    x=alt.X('League:N', title='League'),
    y=alt.Y('PosGroup:N', sort=pos_sort, title='Position group'),
    color=alt.Color(
        'TRB:Q', scale=alt.Scale(scheme='purples'),
        title='Mean TRB / game',
    ),
    tooltip=[
        'League:N', 'PosGroup:N',
        alt.Tooltip('TRB:Q', title='Mean TRB/game', format='.2f'),
    ],
)

labels = alt.Chart(heat_data).mark_text(fontSize=16, fontWeight='bold').encode(
    x='League:N',
    y=alt.Y('PosGroup:N', sort=pos_sort),
    text=alt.Text('TRB:Q', format='.2f'),
    color=alt.condition(
        'datum.TRB > 6', alt.value('white'), alt.value('#222')
    ),
)

chart_heat = (heat + labels).properties(
    width=360, height=320,
    title=alt.TitleParams(
        'Mean Total Rebounds per Game by Position and League',
        subtitle='Darker shade = more rebounds; numeric label flips colour on dark cells',
        anchor='start', fontSize=15,
    ),
).configure_view(stroke=None)

show_altair(chart_heat)

alt.LayerChart(...)

## Playmaking risk — turnovers, minutes, and assists

A **turnover (TOV)** is a possession lost — the ball-handler gives the ball away (bad pass, travel, lost dribble) without producing a shot attempt. Looking at turnovers against two different x-axes tells two related but distinct stories:

- **TOV vs MP (minutes played)** — does the cost of being on the floor grow linearly with playing time, or do top-minute players actually keep their turnover rate flat? A diagonal blob means more minutes = more turnovers; a flat horizontal blob means turnover *rate* is roughly time-independent.
- **TOV vs AST (assists)** — a playmaker risk-reward space. Players further right are creating more for teammates; players higher up are giving the ball away more. The slope of the cluster is essentially the league-wide assist-to-turnover trade-off.

### 11a. Turnovers vs minutes played (with league toggle)

Seaborn-style bivariate kernel density estimate of per-game turnovers against minutes played per game; use the dropdown in the top-right to switch between NBA and WNBA.

In [14]:
def nonzero_tickvals(x_max):
    step = 1 if x_max <= 10 else 5
    return list(np.arange(step, x_max + step / 2, step))

def bivariate_kde_altair(x_col, y_col, x_max, y_max, x_label, y_label, chart_title):
    n_grid = 60
    xgrid = np.linspace(0, x_max, n_grid)
    ygrid = np.linspace(0, y_max, n_grid)
    X, Y = np.meshgrid(xgrid, ygrid)
    grid_pts = np.vstack([X.ravel(), Y.ravel()])
    step_x = xgrid[1] - xgrid[0]
    step_y = ygrid[1] - ygrid[0]

    rows = []
    for league in ['NBA', 'WNBA']:
        sub = combined[combined['League'] == league].dropna(subset=[x_col, y_col])
        kde = gaussian_kde(np.vstack([sub[x_col].values, sub[y_col].values]))
        Z = kde(grid_pts).reshape(X.shape)
        thresh = Z.max() * 0.10
        peak = Z.max()
        for i, xv in enumerate(xgrid):
            for j, yv in enumerate(ygrid):
                if Z[j, i] >= thresh:
                    rows.append({
                        'x_lo': float(xv - step_x / 2),
                        'x_hi': float(xv + step_x / 2),
                        'y_lo': float(yv - step_y / 2),
                        'y_hi': float(yv + step_y / 2),
                        'density_norm': float(Z[j, i] / peak),
                        'League': league,
                    })

    df = pd.DataFrame(rows)

    league_select = alt.selection_point(
        fields=['League'],
        bind=alt.binding_select(options=['NBA', 'WNBA'], name='League  '),
        value=[{'League': 'NBA'}],
    )

    chart = alt.Chart(df).mark_rect().encode(
        x=alt.X('x_lo:Q', title=x_label,
                scale=alt.Scale(domain=[0, x_max]),
                axis=alt.Axis(values=nonzero_tickvals(x_max))),
        x2='x_hi:Q',
        y=alt.Y('y_lo:Q', title=y_label,
                scale=alt.Scale(domain=[0, y_max])),
        y2='y_hi:Q',
        color=alt.Color('League:N', scale=PALETTE_KDE, legend=None),
        opacity=alt.Opacity('density_norm:Q',
                            scale=alt.Scale(domain=[0, 1], range=[0.1, 0.95]),
                            legend=None),
        tooltip=['League:N',
                 alt.Tooltip('density_norm:Q', title='Density (% of peak)', format='.0%')],
    ).add_params(league_select).transform_filter(league_select).properties(
        width=760, height=540,
        title=alt.TitleParams(
            chart_title,
            subtitle=[
                'Smoothed bivariate KDE (density ≥ 10% of peak); use the dropdown to switch league',
                f'Axes fixed at {x_col} ∈ [0, {x_max}], {y_col} ∈ [0, {y_max}]',
            ],
            anchor='start', fontSize=16,
        ),
    ).configure_view(stroke=None)
    return chart

chart_tov_mp = bivariate_kde_altair(
    x_col='MP', y_col='TOV',
    x_max=38, y_max=3.5,
    x_label='Minutes played per game',
    y_label='Turnovers per game',
    chart_title='Turnovers per Game vs Minutes Played',
)
show_altair(chart_tov_mp)

alt.Chart(...)

### 11b. Turnovers vs assists — playmaker risk-reward (with league toggle)

Seaborn-style bivariate kernel density estimate of per-game turnovers against per-game assists; players further right are creating more for teammates, players higher up are losing more possessions. The slope of the cluster reflects the league-wide assist-to-turnover trade-off, a flatter cluster means players can rack up assists without a turnover penalty, while a steeper cluster means high-assist players also turn the ball over a lot. Use the dropdown to switch between leagues.

In [15]:
chart_tov_ast = bivariate_kde_altair(
    x_col='AST', y_col='TOV',
    x_max=7.5, y_max=3.5,
    x_label='Assists per game',
    y_label='Turnovers per game',
    chart_title='Turnovers per Game vs Assists per Game',
)
show_altair(chart_tov_ast)

alt.Chart(...)

### 12. Defensive output vs playing time — multi-series trendlines

A single line chart with four trendlines: one per (league × defensive stat) combination, sharing a common minutes-played axis so the curves can be compared directly. By default the chart shows **non-linear trendlines**, built by binning each series along minutes-played (2.5-minute bins, requiring at least three players per bin), taking the mean stat value in each bin, and drawing a monotone curve through those means. The dropdown switches to the **linear OLS fit** version. Hovering any line surfaces the Pearson `r` (and either the regression equation or the bin count) for that series.

In [16]:
PANEL_LOOKUP = [('NBA', 'STL'), ('NBA', 'BLK'), ('WNBA', 'STL'), ('WNBA', 'BLK')]
STAT_FULL = {'STL': 'Steals', 'BLK': 'Blocks'}
BIN_WIDTH = 2.5

line_rows = []
for league, stat in PANEL_LOOKUP:
    sub = combined[combined['League'] == league].dropna(subset=['MP', stat])
    x_vals = sub['MP'].values
    y_vals = sub[stat].values
    series_name = f'{league} - {STAT_FULL[stat]}'

    slope, intercept = np.polyfit(x_vals, y_vals, 1)
    r_lin = np.corrcoef(x_vals, y_vals)[0, 1]
    x_line = np.linspace(x_vals.min(), x_vals.max(), 500)
    y_line = slope * x_line + intercept
    for x, y in zip(x_line, y_line):
        line_rows.append({
            'MP': float(x), 'Value': float(y),
            'trend_label': series_name, 'League': league, 'Stat': STAT_FULL[stat],
            'Fit': 'Linear',
            'r': r_lin,
            'fit_info': f'y = {slope:.3f}*x + {intercept:.2f}',
        })

    bin_edges = np.arange(np.floor(x_vals.min()), np.ceil(x_vals.max()) + BIN_WIDTH, BIN_WIDTH)
    bin_idx = np.digitize(x_vals, bin_edges) - 1
    bin_centers, bin_means = [], []
    for i in range(len(bin_edges) - 1):
        mask = bin_idx == i
        if mask.sum() >= 3:
            bin_centers.append((bin_edges[i] + bin_edges[i + 1]) / 2)
            bin_means.append(float(y_vals[mask].mean()))

    if len(bin_centers) >= 2:
        bc = np.array(bin_centers)
        bm = np.array(bin_means)
        y_pred = np.interp(x_vals, bc, bm)
        r_nl = float(np.corrcoef(y_vals, y_pred)[0, 1])
        dense_x = np.linspace(bc.min(), bc.max(), 500)
        dense_y = np.interp(dense_x, bc, bm)
        n_bins = len(bin_centers)
    else:
        r_nl = float('nan')
        dense_x, dense_y, n_bins = [], [], 0

    for x, y in zip(dense_x, dense_y):
        line_rows.append({
            'MP': float(x), 'Value': float(y),
            'trend_label': series_name, 'League': league, 'Stat': STAT_FULL[stat],
            'Fit': 'Non-linear (binned means)',
            'r': r_nl,
            'fit_info': f'{n_bins} bins, width {BIN_WIDTH} min',
        })

line_df = pd.DataFrame(line_rows)

TREND_DOMAIN = ['NBA - Steals', 'NBA - Blocks', 'WNBA - Steals', 'WNBA - Blocks']
TREND_COLORS = alt.Scale(
    domain=TREND_DOMAIN,
    range=['#48cae4', '#023e8a', '#ff70a6', '#b5179e'],
)

fit_selection = alt.selection_point(
    fields=['Fit'],
    bind=alt.binding_select(
        options=['Non-linear (binned means)', 'Linear'],
        name='Trend fit  ',
    ),
    value='Non-linear (binned means)',
)

highlight = alt.selection_point(
    fields=['trend_label'],
    on='pointermove',
    empty=False
)

base = alt.Chart(line_df).encode(
    x=alt.X('MP:Q', title='Minutes played per game',
            axis=alt.Axis(values=[5, 10, 15, 20, 25, 30, 35])),
    y=alt.Y('Value:Q', title='Defensive stat per game'),
    color=alt.Color('trend_label:N', scale=TREND_COLORS, title='Series',
                    sort=TREND_DOMAIN),
    tooltip=[
        alt.Tooltip('trend_label:N', title='Series'),
        alt.Tooltip('Fit:N', title='Fit type'),
        alt.Tooltip('r:Q', title='Pearson r', format='.3f'),
        alt.Tooltip('fit_info:N', title='Details'),
    ]
)

lines = base.mark_line(
    interpolate='monotone',
    point=alt.OverlayMarkDef(size=1000, opacity=0, filled=True)
).encode(
    size=alt.condition(highlight, alt.value(4), alt.value(2)),
    opacity=alt.condition(highlight, alt.value(1), alt.value(0.3)),
).add_params(highlight).transform_filter(fit_selection)

chart_def = lines.properties(
    width=900, height=520,
    title=alt.TitleParams(
        'Defensive Output vs Playing Time - Trendlines',
        subtitle='Dropdown switches between non-linear (binned means) and linear OLS fits; hover near any line to highlight it and see its Pearson r',
        anchor='start', fontSize=16,
    ),
).add_params(fit_selection).configure_view(stroke=None)

show_altair(chart_def)

alt.Chart(...)

## Summary


## Reflection
### Visualization 1: Violin plot: Field Goals Attempted per game 
We chose area as the mark to represent the distribution of field goal attempts per game for each league. Position on the y-axis encodes FGA per game, which is a ratio variable with a true zero point. Position is the most accurate channel for quantitative comparisons. Position on the x-axis encodes the league, a nominal variable with no inherent order. The width of the violin shape acts as a size channel, encoding the density of the distribution and showing how many players have a given FGA value. This is what makes the violin plot more informative than a simple box plot. An embedded box plot provides an additional summary of the interquartile range, and a visible mean line further encodes the central tendency using position on the y axis. Color hue distinguishes the two leagues using a qualitative color scheme, which is appropriate because league is a nominal variable with no meaningful midpoint. We thought a sequential or diverging colormap would be incorrect here. An annotated mean label is added as a text mark to make the central tendency immediately readable without requiring the reader to estimate from the shape alone. 

Regarding visual design, the two violins are placed side by side on a shared y axis, following the alignment principle, which enables direct visual comparison of the two distributions. The repeated use of the same violin shape for both leagues follows the repetition principle, keeping the visualization visually consistent. The color difference between the leagues creates contrast, making the NBA–WNBA distinction immediately visible without requiring the reader to consult a legend. 
### Visualization 2: Layered density chart: Points per game
We chose area as the mark, where each filled area represents the smoothed probability
density of points per game for one league. Position on the x axis encodes points per game
(PTS) which is a ratio variable, making position the most accurate channel for this
quantitative comparison. The y axis encodes density, this is also a quantitative variable.
Color hue distinguishes the two leagues using a qualitative scheme which is appropriate
since league is a nominal variable. The opacity of the filled areas is set below 1, allowing the
two distributions to overlap visually without one entirely occluding the other. This is a
deliberate use of the value channel to manage layering. On top of the density areas, dashed
vertical rules mark each league's mean PTS using a line mark. We used the same color hue
encoding for the density curves to maintain visual consistency. The tooltip on the mean rules
encodes the precise mean value as text, giving readers access to the exact figure on hover.

We layered the two density curves on the same axes which follows the alignment principle.
This makes the difference in spread and central tendency directly comparable. The dashed
rule style creates contrast against the solid filled areas, signaling to the reader that these are
summary statistics rather than data distributions. The use of the same color for each
league's density area and its corresponding mean rule follows the repetition principle.

### Visualization 3: Linked scatter + bar: 3PA vs Pointsper game
For the scatter plot we chose point as the mark. Each point represents one individual player.
Position on the x axis encodes three point attempts per game (3PA), and position on the
y axis encodes points per game (PTS), these are both ratio variables which is why position
is the most suitable channel for encoding them accurately. Color hue encodes league
membership using a qualitative scheme, since league is nominal. A rectangular brush
selection (an interactive mark) is added, which filters the bar chart below. Unselected points
are rendered in light gray, using the value (lightness) channel to visually suppress them
without removing them from the plot. This allows the reader to maintain context for the full
dataset while focusing on the selection. The bar chart below uses a bar mark, with position
on the x axis encoding the count of selected players per league, and color hue repeating the
same qualitative league color scheme for visual consistency.

The linked view creates a meaningful interaction by connecting two chart types on the same
dataset. The scatter enables spatial selection, and the bar chart provides an immediate
quantitative summary of the selection. This follows the proximity principle, as both charts are
placed directly adjacent and clearly belong together. The contrast between the highlighted
(colored) and non highlighted (gray) points draws the reader's attention to the selected
subset while preserving the overall distribution as context. The shared color scheme across
both charts follows the repetition principle.

### Visualization 4: Cumulative distribution: Field goal percentage
For the cumulative distribution we chose line as the mark (geometry), where each line traces
the cumulative distribution function (CDF) of field goal percentage for one league. Position
on the x axis encodes FG%, a ratio variable filtered to range 1–99, and position on the y axis
encodes the cumulative share of players at or below a given FG% value. These are both
quantitative ratio variables, making position the best channel for both. The step after
interpolation style is used deliberately, as CDF curves are piecewise step functions. A
smooth interpolation would misrepresent the underlying discrete data. Color hue
distinguishes the two leagues using a qualitative scheme. The tooltip encodes both the exact
FG% value and the corresponding cumulative share as text, giving readers access to
precise values on hover.

By placing both CDF curves on the same axes we followed the alignment principle, allowing
the reader to directly compare how the two leagues' distributions differ at any percentile
threshold. The contrast in color between the two lines makes the gap between the curves
immediately readable, and the steeper or flatter slope of each curve communicates where
the distributions diverge most. We chose this chart type because it allows the reader to ask
"what share of players shoot below X%?" for both leagues simultaneously, which a
histogram or violin plot would make harder to answer precisely

### Visualization 5: Paired donut charts: Three-pointer share
We chose arc as the mark, forming two side by side donut charts. The angular extent (theta)
encodes the proportion of three point attempts versus non three point attempts, a
part to whole ratio. This is why angle was the most natural channel for encoding proportional
composition. An inner radius is used to create the donut shape rather than a full pie, which
reduces the visual weight of the chart and makes the shared legend easier to read without a
center crowding effect. Color hue distinguishes the two shot types (three point vs.
non three point) using a qualitative two category scheme. This is considered appropriate
since shot type is a nominal variable. The two charts are placed side by side using horizontal
concatenation (hconcat), with a shared color scale to ensure direct comparability.

We placed the two donut charts side by side on a shared color scale which follows the
alignment principle, allowing the reader to compare the proportional split between the NBA
and WNBA directly. The consistent use of the same two colors across both charts follows the
repetition principle, preventing confusion about what each color means. A tooltip encodes
the exact count and percentage share on hover, providing precision on top of the visual
estimate. This chart type was chosen specifically because the argument concerns a simple
two part composition (three point share vs. the rest), which a donut chart communicates
more clearly than a bar chart would for a single proportional comparison.

### Visualization 6: Strip plot: Total rebounds per game
We chose point as the mark, where each dot represents one individual player. Position on
the x axis encodes total rebounds per game (TRB) which is a ratio variable. That's why
position is the most accurate channel for this quantitative variable. Position on the y axis
encodes league (NBA / WNBA), a nominal variable. A random jitter offset is applied in the
vertical direction using the yOffset channel, which prevents overplotting among players with
the same or similar TRB values and reveals the full density of the distribution without
smoothing it. Color hue distinguishes the two leagues using a qualitative scheme. Dashed
vertical rules mark the per league mean TRB using the same color hue encoding, adding a
summary statistic on top of the raw data without obscuring the individual points.

We decided to place both leagues on the same x axis to follow the alignment principle,
making the comparison of distributions and means directly readable. The use of the same
mark geometry and color for all players in each league follows the repetition principle. The
contrast between individual dots (light opacity) and the dashed mean rule (solid, higher
visual weight) follows the contrast principle, drawing attention to the summary statistic while
keeping the raw data visible as context. This chart was chosen over a violin or box plot
specifically to avoid smoothing: every player is visible as an individual data point, which
makes outliers and clusters identifiable rather than abstracted away.

### Visualization 7: Bubble chart: Offensive vs defensive rebounds
We chose point as the mark for the bubble chart, where each circle represents one player.
Position on the x axis encodes offensive rebounds per game (ORB), and position on the
y axis encodes defensive rebounds per game (DRB), both ratio variables. Which is why we
chose position as a channel for both. Size encodes total rebounds per game (TRB), using a
scaled size channel (range 20–500 pixels) so that dominant rebounders are visually
prominent. Color hue distinguishes leagues using a qualitative scheme. Opacity is set below
1 to manage overplotting in the denser regions of the scatter.

This is a multivariate visualization that encodes three quantitative variables simultaneously:
ORB, DRB, and TRB. A two axis scatter plot alone wouldn't be able to show this. The size
channel adds a third dimension of information without requiring an additional axis or facet.
The alignment of both leagues on the same axes follows the alignment principle, allowing
direct spatial comparison of where NBA and WNBA rebounders cluster in the ORB–DRB
plane. Color contrast between the leagues enables the reader to immediately see whether
both leagues occupy the same region of the plot or diverge. The tooltip provides individual
player detail on hover, keeping the chart clean at a glance while offering precision on
demand.

### Visualization 8: Dumbbell chart: Mean rebound rate
In the Dumbbell chart we chose point as the mark for the league endpoints, and line as the
mark for the connector between them. Position on the x axis encodes the mean rebounds
per game (either ORB or DRB) which is a ratio variable. Position is the most accurate
channel for quantitative comparison. Position on the y axis encodes the rebound category
(ORB / DRB), a nominal variable. Color hue distinguishes the two leagues using a qualitative
scheme. The connector line between the two endpoints is rendered in light gray, deliberately
given no color encoding, so it functions purely as a visual connector encoding the gap
between the two league means rather than as a data series itself. Text marks display the
exact mean values next to each endpoint, encoding the precise figures as text to allow the
reader to verify the gap without needing to read the axis.

The dumbbell chart was chosen specifically because it encodes the gap between two groups
directly as the length of the connector line, a size channel, rather than requiring the reader to
subtract two bar heights mentally. This follows the principle of using the most perceptually
accurate encoding for the comparison of interest. The alignment of both rebound categories
on the same x axis follows the alignment principle, making it possible to compare the
NBA–WNBA gap for ORB and DRB side by side. The contrast between the large colored
dots and the thin gray connector creates a clear visual hierarchy, drawing the eye to the
group endpoints rather than the connector.

### Visualization 9: Box plot: Offensive rebound share
We chose point as the mark for the individual player dots overlaid on the box plot, and a
composite box mark (whiskers, IQR box, and median line) for the statistical summary.
Position on the y axis encodes offensive rebound share (ORB / TRB), this is a ratio variable.
position is the most accurate channel for this quantitative comparison. Position on the x axis
encodes league (NBA / WNBA), a nominal variable. Color hue distinguishes the leagues
using a qualitative scheme. The y axis is formatted as a percentage, matching the
interpretive meaning of the variable (share out of 1.0). An annotated mean label is added as
a text mark to make the average offensive rebound share immediately readable alongside
the median shown by the box plot.

Regarding visual design, the box plot was chosen because ORB share is a rate variable
(values between 0 and 1) for which the median, interquartile range, and outlier spread are
more informative than a simple mean comparison. Overlaying the raw player dots follows the
contrast principle: the individual data points provide granular context while the box plot
provides a compact statistical summary in the foreground. The alignment of both leagues on
the same y axis follows the alignment principle, enabling direct comparison. The repetition of
the same color scheme used throughout the data story reinforces the visual identity of each
league.

### Visualization 10: Heatmap: Mean rebounds by position and league
With rect as a mark we formed a grid where each cell represents one combination of position
group and league. Color value (lightness within the "purples" sequential colormap) encodes
mean total rebounds per game, a ratio variable. A sequential colormap is appropriate here
because there is no meaningful midpoint in rebound rate. The values range from low to high
without a natural no contrast center. Text marks overlay each cell with the precise mean
value, and the text color flips automatically from dark to light when the cell background
exceeds a threshold, using the contrast principle to ensure legibility across all shades. The
y axis encodes position group (Guard, Forward, Hybrid, Center), a nominal variable ordered
by basketball convention (perimeter to interior players). The x-axis encodes league, a
nominal variable.

Regarding visual design, the heatmap was chosen because it enables simultaneous
comparison of two categorical variables (position group and league) against one quantitative
variable (mean TRB), which would require multiple separate charts if done with bar or line
charts. The sequential colormap communicates the quantitative magnitude of rebounding
directly through visual intensity, following the principle that color value is well suited for
encoding ordered quantitative differences. The numeric labels inside each cell follow the
contrast principle and ensure that readers who want precise values are not limited to reading
the color shade alone.

### Visualization 11a: Bivariate KDE: Turnovers vs minutes played
We chose rect as the mark for the bivariate KDE where each cell in a 60×60 grid represents
a small region of the "turnovers by minutes played" space. Color value (lightness within a
sequential colormap per league) encodes the normalized kernel density estimate at each
grid location, a quantitative variable. We chose to make color value the encoding channel. A
sequential colormap is appropriate because density has no meaningful midpoint: it ranges
from near zero (sparse region) to peak (densest cluster of players). Only grid cells above
10% of the peak density are rendered, with all others suppressed, to focus the visualization
on the meaningful region of the distribution and reduce visual noise. A dropdown binding
allows the reader to switch between NBA and WNBA, making the interaction meaningful: the
two leagues cannot be displayed simultaneously as overlapping KDE grids without creating
unreadable color mixing, so the toggle enables direct comparison while keeping each view
interpretable on its own.

The bivariate KDE was chosen over a scatter plot because the large number of players in
the combined dataset creates severe overplotting, which obscures the shape of the joint
distribution. The KDE smooths this into a continuous density surface that reveals where
most players cluster in the TOV–MP space. The contrast between the dense central region
(dark) and the sparse periphery (light) makes the location and spread of the distribution
immediately readable. The dropdown toggle creates meaningful interactivity: the reader
actively switches between leagues and can note whether the clusters shift, widen, or overlap
differently.


### Visualization 12: Multi-series trendlines: Defensive output vs playing time
We chose line as the mark, where each line traces the trendline for one (league × defensive
stat) combination. There are four series in total (NBA steals, NBA blocks, WNBA steals,
WNBA blocks). Position on the x axis encodes minutes played per game (MP), a ratio
variable. Position on the y axis encodes the defensive stat value per game, also a ratio
variable. Color hue distinguishes the four series using a qualitative four color scheme,
appropriate because the series label is a nominal variable (each series is a distinct
league–stat combination with no meaningful order). Two trendline types are available: a
non-linear binned means fit (default) and a linear OLS fit, switchable via a dropdown. This is
a meaningful interaction because it allows the reader to evaluate whether the relationship
between playing time and defensive output is better captured by a linear or a non-linear
model. This is a question that cannot be answered from a single static view. Hover
interaction highlights the closest line and shows the Pearson r and fit equation in a tooltip,
providing analytical detail on demand without cluttering the chart.

Overlaying all four series on a single shared axis follows the alignment principle, making
cross league and cross stat comparisons readable in one glance. The opacity and
stroke width encoding (full opacity and 4px for the hovered line, reduced opacity and 2px for
all others) follows the contrast principle, drawing the reader's attention to the line they are
exploring while keeping the others as context. The monotone interpolation style for the
non-linear fit produces smooth curves without overshooting, which is more honest than a
polynomial fit when the binned data has local variability. The dropdown and hover together
constitute a genuinely meaningful interactive visualization, as described in the course
requirements.

### Peer review and feedback